In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType,
)

# Initialize Spark with Kafka package (adjust package version to match your Spark version if needed)
spark = (
    SparkSession.builder.appName("BigData_Lab2")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,org.apache.kafka:kafka-clients:3.6.0",
    )
    .master("local[*]")
    .getOrCreate()
)

# --- 1. DEFINE SCHEMAS ---
movies_schema = StructType(
    [
        StructField("movieId", IntegerType(), True),
        StructField("title", StringType(), True),
        StructField("genres", StringType(), True),
    ]
)

ratings_schema = StructType(
    [
        StructField("userId", DoubleType(), True),
        StructField("movieId", DoubleType(), True),
        StructField("rating", DoubleType(), True),
        StructField("timestamp", DoubleType(), True),
    ]
)

# Tags stays the same, it already outputs clean integers
tags_schema = StructType(
    [
        StructField("userId", IntegerType(), True),
        StructField("movieId", IntegerType(), True),
        StructField("tag", StringType(), True),
        StructField("timestamp", IntegerType(), True),
    ]
)

# --- 2. INGEST FROM KAFKA & APPLY SCHEMAS ---

df_movies_raw = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "Lab1_movies")
    .option("startingOffsets", "earliest")
    .load()
)

# Convert binary 'value' to string, parse JSON, and expand columns
df_movies = (
    df_movies_raw.selectExpr("CAST(value AS STRING)")
    .select(from_json(col("value"), movies_schema).alias("data"))
    .select("data.*")
)

# Ratings: Parse as Double, then cast IDs to Integer and epoch to Timestamp
df_ratings_raw = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "Lab1_ratings")
    .option("maxOffsetsPerTrigger", 100)
    # .option("startingOffsets", "earliest")
    .load()
)

df_ratings = (
    df_ratings_raw.selectExpr("CAST(value AS STRING)")
    .select(from_json(col("value"), ratings_schema).alias("data"))
    .select("data.*")
    .withColumn("userId", col("userId").cast("integer"))
    .withColumn("movieId", col("movieId").cast("integer"))
    .withColumn("timestamp", col("timestamp").cast("timestamp"))
)  # Converts epoch seconds to YYYY-MM-DD HH:MM:SS

# Tags: Parse as Integer, then cast epoch to Timestamp
df_tags_raw = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "Lab1_tags")
    .option("maxOffsetsPerTrigger", 100)
    .load()
)

df_tags = (
    df_tags_raw.selectExpr("CAST(value AS STRING)")
    .select(from_json(col("value"), tags_schema).alias("data"))
    .select("data.*")
    .withColumn("timestamp", col("timestamp").cast("timestamp"))
)  # Converts epoch seconds to YYYY-MM-DD HH:MM:SS

print("Starting the Ratings stream to console...")

# x = 420
# # query movies with movie id is x
# query_movies = (
#     df_movies.filter(col("movieId") == x)
#     .writeStream.outputMode("append")
#     .format("console")
#     .option("truncate", "false")
#     .start()
# )

# # query ratings with movie id is x
# query_ratings = (
#     df_ratings.filter(col("movieId") == x)
#     .writeStream.outputMode("append")
#     .format("console")
#     .option("truncate", "false")
#     .start()
# )

# # Define the sink (console) and start the stream
# query_ratings = df_ratings.writeStream \
#     .outputMode("append") \
#     .format("console") \
#     .option("truncate", "false") \
#     .start()
# query_tags = df_tags.writeStream \
#     .outputMode("append") \
#     .format("console") \
#     .option("truncate", "false") \
#     .start()

# # This is the crucial part: it keeps the Python script running forever
# # to continuously listen for new Kafka messages.
# query_ratings.awaitTermination()
# query_tags.awaitTermination()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/28 22:28:19 WARN Utils: Your hostname, LAPTOP-2TOMM4HG, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/28 22:28:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/liemak363/repo/Big%20Data%20Lab/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/liemak363/.ivy2.5.2/cache
The jars for the packages stored in: /home/liemak363/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.apache.kafka#kafka-clients added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0a41abb4-c1a6-4cbd-98a4-075a257bd9f0;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13

Starting the Ratings stream to console...


In [2]:
from pyspark.sql.functions import col, split, explode, count
import os

# 1. Load the STATIC movies DataFrame (adjust path if needed)
dataset_path = os.path.expanduser(
    "~/.cache/kagglehub/datasets/grouplens/movielens-latest-small/versions/2"
)
df_movies_static_ex2 = spark.read.csv(
    f"{dataset_path}/movies.csv", header=True, inferSchema=True
)

# 2. Join the STREAMING ratings (from Ex 1) with the STATIC movies
df_joined_ex2 = df_ratings.join(df_movies_static_ex2, on="movieId", how="inner")

# 3. Transform: Split the string by the pipe character and explode it
# Note: We use r"\|" because the pipe is a special regex character and must be escaped
df_exploded_ex2 = df_joined_ex2.withColumn(
    "single_genre", explode(split(col("genres"), r"\|"))
)

# 4. Aggregate: Group by the new flattened genre column, count, and THEN sort
df_trending_genres_ex2 = (
    df_exploded_ex2.groupBy("single_genre")
    .agg(count("*").alias("total_ratings"))
    .orderBy(col("total_ratings").desc())
)

# 5. Output: Write to console every 5 seconds in Complete mode
query_ex2 = (
    df_trending_genres_ex2.writeStream.outputMode("complete")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="5 seconds")
    .start()
)

# Keep the cell running
query_ex2.awaitTermination()

26/04/28 22:28:33 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-176175cc-4fe8-4103-8caa-63d1fa67f680. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/28 22:28:33 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/28 22:28:33 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------+-------------+
|single_genre|total_ratings|
+------------+-------------+
+------------+-------------+



26/04/28 22:28:42 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 9510 milliseconds
26/04/28 22:29:01 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 6654 milliseconds


-------------------------------------------
Batch: 1
-------------------------------------------
+------------+-------------+
|single_genre|total_ratings|
+------------+-------------+
|Drama       |35           |
|Romance     |26           |
|Comedy      |26           |
|Action      |17           |
|Thriller    |15           |
|Adventure   |14           |
|Children    |9            |
|Crime       |8            |
|Fantasy     |5            |
|Sci-Fi      |5            |
|Animation   |4            |
|Western     |4            |
|War         |3            |
|Mystery     |3            |
|Musical     |3            |
|IMAX        |2            |
|Film-Noir   |1            |
|Horror      |1            |
+------------+-------------+



26/04/28 22:29:09 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 7348 milliseconds


...

26/04/28 22:30:30 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 5494 milliseconds


-------------------------------------------
Batch: 17
-------------------------------------------
+------------+-------------+
|single_genre|total_ratings|
+------------+-------------+
|Drama       |699          |
|Comedy      |642          |
|Action      |525          |
|Thriller    |518          |
|Adventure   |390          |
|Romance     |346          |
|Crime       |308          |
|Sci-Fi      |196          |
|Children    |189          |
|Fantasy     |150          |
|Animation   |104          |
|War         |86           |
|Horror      |84           |
|Mystery     |81           |
|Western     |81           |
|Musical     |78           |
|IMAX        |56           |
|Documentary |7            |
|Film-Noir   |2            |
+------------+-------------+



-------------------------------------------
Batch: 18
-------------------------------------------
+------------+-------------+
|single_genre|total_ratings|
+------------+-------------+
|Drama       |742          |
|Comedy      |684          |
|Action      |552          |
|Thriller    |545          |
|Adventure   |414          |
|Romance     |374          |
|Crime       |321          |
|Sci-Fi      |204          |
|Children    |200          |
|Fantasy     |157          |
|Animation   |113          |
|Horror      |90           |
|War         |89           |
|Mystery     |88           |
|Western     |88           |
|Musical     |84           |
|IMAX        |58           |
|Documentary |7            |
|Film-Noir   |2            |
+------------+-------------+



26/04/28 22:30:35 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 5224 milliseconds
26/04/28 22:30:42 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 6322 milliseconds


-------------------------------------------
Batch: 19
-------------------------------------------
+------------+-------------+
|single_genre|total_ratings|
+------------+-------------+
|Drama       |765          |
|Comedy      |707          |
|Action      |572          |
|Thriller    |570          |
|Adventure   |431          |
|Romance     |382          |
|Crime       |337          |
|Sci-Fi      |211          |
|Children    |206          |
|Fantasy     |161          |
|Animation   |117          |
|Horror      |93           |
|War         |92           |
|Mystery     |90           |
|Western     |89           |
|Musical     |87           |
|IMAX        |61           |
|Documentary |7            |
|Film-Noir   |2            |
+------------+-------------+

